# Activity 2: Machine Learning Anomaly Detection for Time Series Data

## Introduction

In **Activity 1**, you learned about fundamental anomaly detection algorithms like KNN, LOF, Isolation Forest, and AutoEncoders. These algorithms work great for detecting anomalies in general datasets.

However, **time series data presents unique challenges**:
- Values change over time with trends and patterns
- Seasonal patterns can mask true anomalies
- Context matters: what's normal on Tuesday might be anomalous on Friday
- We need to consider temporal relationships between observations

**This activity focuses on how to APPLY machine learning algorithms to time series** by using different temporal preprocessing techniques. You'll see that **the SAME algorithm can find DIFFERENT anomalies** depending on how you prepare the time series data!

---

## Learning Objectives

By the end of this activity, you will be able to:

1. **Apply 4 different temporal preprocessing approaches** to time series anomaly detection:
   - Point Outlier Detection (raw values)
   - Decomposition-Based Detection (residuals)
   - Sliding Window Detection (temporal patterns)
   - Feature Engineering Detection (contextual features)

2. **Understand when each approach is appropriate:**
   - Point anomalies vs. pattern anomalies
   - How seasonality affects detection
   - Trade-offs between different approaches

3. **Evaluate anomaly detection results** against known ground truth events

4. **Compare and contrast** how the same algorithm behaves with different preprocessing

---

## Key Insight

**The SAME algorithm (e.g., KNN or LOF) will detect DIFFERENT anomalies depending on how you preprocess the time series data!**

- **Raw values**: Finds extreme high/low points
- **Residuals**: Finds deviations from expected seasonal patterns
- **Sliding windows**: Finds unusual temporal sequences
- **Feature engineering**: Finds contextual anomalies (e.g., unusual for a Monday)

Understanding these differences is crucial for real-world time series anomaly detection!

---

# Setup and Data Loading

## Technical Requirements

In [ ]:
import matplotlib 
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 150

In [ ]:
from pyod import version
version.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# PyOD algorithms
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.iforest import IForest

# Time series tools
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

## Dataset: NYC Taxi Ridership

We'll use NYC taxi ridership data, which contains the number of passengers over time. This is an excellent dataset for learning anomaly detection because:

- It has **clear seasonal patterns** (daily, weekly, yearly)
- It contains **known anomalies** (holidays, special events)
- It's a **real-world use case** (anomaly detection in transportation systems)

The dataset has observations every 30 minutes, showing how many taxi passengers were recorded during each time period.

**For this activity**, we'll resample to **daily** frequency to focus on high-level patterns and known events.

In [ ]:
# Load data
file = Path("data/nyc_taxi.csv")
nyc_taxi = pd.read_csv(file, 
                     index_col='timestamp', 
                     parse_dates=True)
nyc_taxi.index.freq = '30min'

# Resample to daily for this activity
tx = nyc_taxi.resample('D').mean()
print(f"Dataset shape: {tx.shape}")
print(f"Date range: {tx.index[0]} to {tx.index[-1]}")
tx.head()

## Helper Functions

These functions will help us throughout the activity. **Read through them to understand what they do!**

In [ ]:
def plot_outliers(outliers, data, method='KNN', halignment='right', valignment='bottom', labels=False):
    """
    Plot time series data with highlighted outliers.
    
    Parameters
    ----------
    outliers : pandas.DataFrame or pandas.Series
        The DataFrame or Series containing the outlier data points.
    data : pandas.DataFrame or pandas.Series
        The complete time series data.
    method : str, default='KNN'
        The outlier detection method used, displayed in the plot title.
    halignment : str, default='right'
        Horizontal alignment for the date labels ('left', 'center', or 'right').
    valignment : str, default='bottom'
        Vertical alignment for the date labels ('top', 'center', or 'bottom').
    labels : bool, default=False
        If True, displays date labels for each outlier point.
        
    Returns
    -------
    None
        The function shows the plot but does not return any value.
    """
    
    fig, ax = plt.subplots(figsize=(10, 6))
        
    data.plot(ax=ax, alpha=0.6)
    
    # Plot outliers
    if labels:
        outliers.plot(ax=ax, style='rx', markersize=8, legend=False)
        
        # Add text labels for each outlier
        for idx, value in outliers['value'].items():
            ax.text(idx, value, f'{idx.date()}', 
                   horizontalalignment=halignment, 
                   verticalalignment=valignment)
    else:
        outliers.plot(ax=ax, style='rx', legend=False)
    
    ax.set_title(f'NYC Taxi - {method}')
    ax.set_xlabel('date')
    ax.set_ylabel('# of passengers')
    ax.legend(['nyc taxi', 'outliers'])
    
    plt.tight_layout()
    plt.show()

In [ ]:
def create_sliding_windows(df, window_size):
    """Transform time series data into sliding windows for anomaly detection.
    
    Creates a DataFrame where each row represents a sliding window of observations,
    allowing anomaly detection algorithms to identify unusual temporal patterns.
    
    Args:
        df (pd.DataFrame): Univariate time series data with values in a single column
        window_size (int): Number of time steps in each sliding window
        
    Returns:
        pd.DataFrame: DataFrame where each row is a complete window of observations,
                     and columns represent the position within the window
    """
    # Validate input
    if not isinstance(window_size, int) or window_size < 1:
        raise ValueError("Window size must be a positive integer")
    
    # Convert DataFrame to 1D array
    d = df.values.squeeze()
    
    # Create sliding windows using numpy's efficient implementation
    windows = np.lib.stride_tricks.sliding_window_view(d, window_shape=window_size)[:-1]
    
    # Create column names for positions within the window
    cols = [f'pos_{i}' for i in range(window_size)]
    
    # Create DataFrame with windows
    windows_df = pd.DataFrame(windows, columns=cols, index=df.index[window_size:])
    
    return windows_df

In [ ]:
# Install holidays package if needed
!uv pip install holidays

In [ ]:
import holidays

def add_time_features(df):
    """Add time-based and exogenous features to a daily time series dataset.
    
    Creates features useful for time series anomaly detection by encoding 
    temporal patterns and external factors that might influence the data.
    
    Args:
        df (pd.DataFrame): Time series DataFrame with DatetimeIndex
        
    Returns:
        pd.DataFrame: Original DataFrame with additional time-based features
    """
    # Create a copy to avoid modifying the original
    result = df.copy()
    
    # Cyclical encoding for day of week (weekly seasonality)
    result['dow_sin'] = np.sin(2 * np.pi * result.index.dayofweek / 7)
    result['dow_cos'] = np.cos(2 * np.pi * result.index.dayofweek / 7)
    
    # Cyclical encoding for month (yearly seasonality)
    result['month_sin'] = np.sin(2 * np.pi * result.index.month / 12)
    result['month_cos'] = np.cos(2 * np.pi * result.index.month / 12)
    
    # Keep year for trend analysis
    result['year'] = result.index.year
    
    # Trend feature (simple incremental counter)
    result['time'] = np.arange(1, len(result)+1)
    
    # US holidays - fix the datetime comparison warning
    us_holidays = holidays.US(years=result.index.year.unique())
    result['is_holiday'] = result.index.map(lambda x: x in us_holidays).astype(int)
    
    # Weekend feature
    result['is_weekend'] = (result.index.dayofweek >= 5).astype(int)
    
    # Month start/end features (can be important for taxi data)
    result['is_month_start'] = result.index.is_month_start.astype(int)
    result['is_month_end'] = result.index.is_month_end.astype(int)
    
    # For NYC taxi data: check if it's a typical commuting day
    result['is_commuting_day'] = ((~result['is_holiday'].astype(bool)) & 
                                 (~result['is_weekend'].astype(bool))).astype(int)
    
    return result

## Visualize the Data and Known Anomalies

Before applying any algorithms, let's visualize the data and identify some **known anomalies**. These known anomalies will serve as our ground truth to evaluate how well our algorithms perform.

**Known Anomaly Dates:**
- **November 1, 2014**: NYC Marathon
- **November 27, 2014**: Thanksgiving
- **December 25, 2014**: Christmas
- **January 1, 2015**: New Year's Day
- **January 27, 2015**: Major snowstorm (Blizzard)

These dates should show unusual taxi ridership patterns!

In [ ]:
# Define known anomaly dates
nyc_dates = [
    "2014-11-01",  # NYC Marathon
    "2014-11-27",  # Thanksgiving
    "2014-12-25",  # Christmas
    "2015-01-01",  # New Year's Day
    "2015-01-27"   # Blizzard
]

known_outliers = tx.loc[nyc_dates]
print("Known Anomaly Dates and Values:")
print(known_outliers)

In [ ]:
# Visualize known anomalies
plot_outliers(known_outliers, tx, 'Known Outliers', labels=True)

---

# Part 1: Detecting Point Outliers (Raw Values)

## Concept

In this approach, we apply anomaly detection algorithms **directly to the raw time series values**. This detects **point anomalies** - individual observations that have unusual values compared to other observations, **without considering temporal context or trends**.

**What does this find?**
- Days with extremely high or low ridership
- Simple outliers based on value magnitude
- Does NOT account for trends, seasonality, or day-of-week patterns

**Example:**
- If most days have 10,000-15,000 passengers
- A day with 5,000 passengers would be detected as an anomaly
- Even if it's Christmas (expected to be low!)

## TODO 1.1: Apply KNN to Raw Time Series Values

**Your Task:**
1. Initialize a KNN detector with contamination=0.03 and n_neighbors=5
2. Fit the detector on the raw time series data (`tx`)
3. Get predictions and extract the outliers
4. Print the number of outliers detected
5. Print the outlier dates and values

**Hint:** Follow the PyOD workflow: initialize -> fit -> predict -> extract outliers

In [ ]:
# TODO: Initialize KNN detector
knn = None  # YOUR CODE HERE

# TODO: Fit on raw time series data
# YOUR CODE HERE

# TODO: Get predictions
knn_pred = None  # YOUR CODE HERE (convert to Series with tx.index)

# TODO: Extract outliers (where prediction == 1)
knn_outliers = None  # YOUR CODE HERE

# TODO: Print results
# YOUR CODE HERE

In [ ]:
# Visualize KNN results
plot_outliers(knn_outliers, tx, 'KNN - Point Outliers', labels=True)

## TODO 1.2: Apply LOF to Raw Time Series Values

**Your Task:**
1. Initialize a LOF detector with contamination=0.03 and n_neighbors=20
2. Fit, predict, and extract outliers
3. Print and visualize results

In [ ]:
# TODO: Apply LOF to raw time series
# YOUR CODE HERE

lof = None  # YOUR CODE HERE

# Fit, predict, extract outliers
# YOUR CODE HERE

lof_outliers = None  # YOUR CODE HERE

In [ ]:
# Visualize LOF results
plot_outliers(lof_outliers, tx, 'LOF - Point Outliers', labels=True)

## Exercise 1.3: Compare KNN vs LOF Results

**Questions to answer:**

1. How many outliers did each algorithm detect?
2. Do they detect the same dates or different dates?
3. Which known anomalies (NYC Marathon, Thanksgiving, Christmas, New Year, Blizzard) did each algorithm detect?
4. Why might KNN and LOF find different anomalies even on the same raw data?

**Your Analysis:**

In [ ]:
# TODO: Compare results
# Hint: Print the outlier dates from both algorithms side by side
# Compare with known_outliers

print("Known Anomalies:")
print(nyc_dates)
print("\nKNN Detected:")
# YOUR CODE HERE
print("\nLOF Detected:")
# YOUR CODE HERE

**Write your analysis here:**

1. Number of outliers:
   - KNN: [YOUR ANSWER]
   - LOF: [YOUR ANSWER]

2. Are they the same or different?
   - [YOUR ANSWER]

3. Which known anomalies were detected?
   - [YOUR ANSWER]

4. Why do they differ?
   - [YOUR ANSWER - Think about how KNN uses average distance vs how LOF uses local density]

---

# Part 2: Detecting Outliers After Time Series Decomposition

## Concept: Why Decompose Time Series?

When detecting anomalies in time series, we often face a challenge: **seasonal patterns can mask true anomalies**. For example:
- Low taxi ridership on Christmas is **normal** (expected pattern)
- Low taxi ridership on a regular Tuesday is **anomalous** (unexpected)

**Time Series Decomposition** separates the data into components:
1. **Trend**: Long-term increase or decrease
2. **Seasonal**: Repeating patterns (daily, weekly, yearly)
3. **Residual**: What's left after removing trend and seasonality

### The Key Insight

By applying anomaly detection to the **residuals** (not raw values), we detect:
- Points that deviate from expected seasonal patterns
- Anomalies that are unusual **given the context** (day of week, season, etc.)
- More meaningful anomalies for time series data

**Example:**
- Raw value: Christmas has low ridership (detected as anomaly)
- Residual: Christmas ridership is **as expected for a holiday** (not an anomaly)
- The blizzard has **unexpectedly low** ridership even for winter (true anomaly!)

## TODO 2.1: Decompose Time Series and Visualize Residuals

**Your Task:**
1. Use STL (Seasonal and Trend decomposition using Loess) to decompose the time series
2. Extract the residuals
3. Visualize the residuals to understand what they represent

**Hint:** Use `STL(tx, seasonal=7)` for weekly seasonality (7 days)

In [ ]:
# TODO: Decompose time series using STL
stl = None  # YOUR CODE HERE: STL(tx, seasonal=7)
result = None  # YOUR CODE HERE: fit the STL

# TODO: Extract residuals
residuals = None  # YOUR CODE HERE: get residuals and convert to DataFrame

# Visualize residuals
residuals.plot(title='Residuals after STL decomposition', figsize=(10, 4))
plt.ylabel('Residual value')
plt.show()

## TODO 2.2: Apply KNN to Residuals

**Your Task:**
1. Initialize a KNN detector
2. Fit it on the **residuals** (not raw values!)
3. Get predictions and extract outliers
4. Map the outlier dates back to the original time series for visualization

In [ ]:
# TODO: Apply KNN to residuals
knn_decomp = None  # YOUR CODE HERE

# TODO: Fit and predict
# YOUR CODE HERE

knn_pred_r = None  # YOUR CODE HERE: predictions as Series

# TODO: Extract outliers and map to original time series
knn_outliers_r = None  # YOUR CODE HERE: get outlier dates and values from tx

print(f"Number of KNN outliers (decomposition): {knn_pred_r.sum()}")
print("\nKNN Outliers (decomposition):")
print(knn_outliers_r)

In [ ]:
# Visualize on original time series
plot_outliers(knn_outliers_r, tx, 'KNN on Residuals (Decomposition)', labels=True)

## TODO 2.3: Apply LOF to Residuals

Repeat the same process with LOF.

In [ ]:
# TODO: Apply LOF to residuals
# YOUR CODE HERE

lof_decomp = None  # YOUR CODE HERE
# Fit, predict, extract outliers
# YOUR CODE HERE

lof_outliers_r = None  # YOUR CODE HERE

In [ ]:
# Visualize LOF results on residuals
plot_outliers(lof_outliers_r, tx, 'LOF on Residuals (Decomposition)', labels=True)

## Exercise 2.4: Why Are Residuals Better?

**Questions to answer:**

1. Compare the outliers detected on raw values (Part 1) vs residuals (Part 2). What changed?
2. Which approach detected more of the known anomalies?
3. Why do residuals give better results for time series anomaly detection?
4. When would you prefer raw values over residuals?

**Your Analysis:**

**Write your analysis here:**

1. What changed between raw values and residuals?
   - [YOUR ANSWER]

2. Which approach detected more known anomalies?
   - Raw values: [YOUR ANSWER]
   - Residuals: [YOUR ANSWER]

3. Why are residuals better?
   - [YOUR ANSWER - Think about seasonal patterns and expected vs unexpected deviations]

4. When would you prefer raw values?
   - [YOUR ANSWER - Consider cases where magnitude matters more than seasonal context]

---

# Part 3: Detecting Contextual Outliers with Sliding Windows

## Concept: What are Contextual Outliers?

A **contextual outlier** is a data point that's anomalous **in its specific context** but might be normal in a different context. For time series:
- A value might be normal by itself
- But **unusual given recent history**

**Example:** A taxi count of 10,000 might be normal, but if the previous 6 days all had 30,000, then 10,000 becomes anomalous.

### The Sliding Window Approach

Instead of looking at individual values, we create **windows of consecutive observations**:

1. **Create windows**: Each row contains the last N consecutive values
2. **Apply detection**: The algorithm sees entire patterns, not just points
3. **Detect pattern anomalies**: Unusual sequences or temporal patterns

**What does this detect?**
- A sudden drop after a week of high values
- An unusual temporal pattern (e.g., increasing when it should decrease)
- Context-dependent anomalies

**Example Window (size=7):**
```
Date: 2014-07-07
Window: [12000, 13000, 12500, 13200, 12800, 13100, 5000]
                                                    ^^^^^ Anomalous in context!
```

## TODO 3.1: Create Sliding Windows

**Your Task:**
1. Use the `create_sliding_windows()` function to create 7-day windows
2. Examine the shape and structure of the windowed data
3. Print a few examples to understand what the windows look like

In [ ]:
# TODO: Create sliding windows of size 7
window_size = 7
tx_sw = None  # YOUR CODE HERE: use create_sliding_windows()

print(f"Original data shape: {tx.shape}")
print(f"Windowed data shape: {tx_sw.shape}")
print(f"\nEach row now contains {window_size} consecutive days")
print("\nFirst few windows:")
print(tx_sw.head(3))

## TODO 3.2: Apply KNN to Sliding Windows

**Your Task:**
1. Apply KNN to the windowed data
2. Extract and visualize outliers
3. Compare with previous approaches

In [ ]:
# TODO: Apply KNN to sliding windows
knn_sw = None  # YOUR CODE HERE

# TODO: Fit and predict
# YOUR CODE HERE

knn_pred_sw = None  # YOUR CODE HERE

# TODO: Extract outliers and map to original time series
knn_outliers_sw = None  # YOUR CODE HERE

print(f"Number of KNN outliers (sliding window): {knn_pred_sw.sum()}")
print("\nKNN Outliers (sliding window):")
print(knn_outliers_sw)

In [ ]:
# Visualize KNN sliding window results
plot_outliers(knn_outliers_sw, tx, 'KNN - Sliding Window', labels=True)

## TODO 3.3: Apply LOF to Sliding Windows

In [ ]:
# TODO: Apply LOF to sliding windows
# YOUR CODE HERE

lof_sw = None  # YOUR CODE HERE
# Fit, predict, extract outliers

lof_outliers_sw = None  # YOUR CODE HERE

In [ ]:
# Visualize LOF sliding window results
plot_outliers(lof_outliers_sw, tx, 'LOF - Sliding Window', labels=True)

## Exercise 3.4: Interpret Pattern Anomalies

**Questions to answer:**

1. What is a "pattern anomaly" vs a "point anomaly"?
2. How do the sliding window results differ from the point outlier results (Part 1)?
3. Do sliding windows detect different types of events compared to residuals (Part 2)?
4. What does it mean when a date is flagged as an anomaly using sliding windows?

**Your Analysis:**

**Write your analysis here:**

1. Pattern anomaly vs point anomaly:
   - Point anomaly: [YOUR ANSWER]
   - Pattern anomaly: [YOUR ANSWER]

2. How do sliding window results differ from point outliers?
   - [YOUR ANSWER]

3. Sliding windows vs residuals:
   - [YOUR ANSWER - Consider what each approach captures]

4. What does it mean when sliding windows flag a date?
   - [YOUR ANSWER - Think about the 7-day sequence leading up to that date]

---

# Part 4: Detecting Contextual Outliers with Feature Engineering

## Concept: The Power of Feature Engineering

While sliding windows capture recent history, **feature engineering** adds explicit **contextual information**:

- **Temporal features**: Day of week, month, year, time trends
- **Cyclical encoding**: Sine/cosine transformations for periodic patterns
- **Domain knowledge**: Holidays, weekends, special events

### Why Use Both Sliding Windows AND Features?

Combining both approaches gives us the best of both worlds:

1. **Sliding windows**: Capture raw sequential patterns ("what happened recently?")
2. **Engineered features**: Add context ("what day is it? is it a holiday?")

**Example:**
- Without features: "This is different from the last 7 days"
- With features: "This is different from the last 7 days, AND it's a Monday, AND it's a holiday"

The algorithm can now learn:
- "Mondays usually have higher ridership"
- "Holidays usually have lower ridership"
- "A Monday with holiday-level ridership is unusual!"

## TODO 4.1: Create Temporal Features

**Your Task:**
1. Use `add_time_features()` to create temporal features
2. Examine what features were created
3. Understand how cyclical encoding works

In [ ]:
# TODO: Create temporal features
features = None  # YOUR CODE HERE: use add_time_features()

print("Original columns:", tx.columns.tolist())
print("\nColumns after feature engineering:", features.columns.tolist())
print("\nFirst few rows with features:")
print(features.head())

## TODO 4.2: Combine Sliding Windows with Features

**Your Task:**
1. Create sliding windows (7 days)
2. Add temporal features
3. Combine both using `pd.merge()`
4. Examine the resulting rich feature set

In [ ]:
# TODO: Create sliding windows
windows = None  # YOUR CODE HERE

# TODO: Create features
features = None  # YOUR CODE HERE

# TODO: Combine windows with features
# Hint: Use pd.merge() and drop the 'value' column from features to avoid duplication
combined = None  # YOUR CODE HERE

print(f"Windows shape: {windows.shape}")
print(f"Features shape: {features.shape}")
print(f"Combined shape: {combined.shape}")
print(f"\nCombined columns: {combined.columns.tolist()}")

## TODO 4.3: Apply KNN to Combined Features

**Your Task:**
1. Apply KNN to the combined feature set
2. Extract and visualize outliers
3. Compare with previous approaches

In [ ]:
# TODO: Apply KNN to combined features
knn_fe = None  # YOUR CODE HERE

# TODO: Fit and predict
# YOUR CODE HERE

knn_pred_fe = None  # YOUR CODE HERE

# TODO: Extract outliers
knn_outliers_fe = None  # YOUR CODE HERE

print(f"Number of KNN outliers (feature engineering): {knn_pred_fe.sum()}")
print("\nKNN Outliers (feature engineering):")
print(knn_outliers_fe)

In [ ]:
# Visualize KNN feature engineering results
plot_outliers(knn_outliers_fe, tx, 'KNN - Sliding Window + Features', labels=True)

## TODO 4.4: Apply LOF to Combined Features

In [ ]:
# TODO: Apply LOF to combined features
# YOUR CODE HERE

lof_fe = None  # YOUR CODE HERE
# Fit, predict, extract outliers

lof_outliers_fe = None  # YOUR CODE HERE

In [ ]:
# Visualize LOF feature engineering results
plot_outliers(lof_outliers_fe, tx, 'LOF - Sliding Window + Features', labels=True)

## Exercise 4.5: Which Features Matter Most?

**Questions to investigate:**

1. How do the feature engineering results compare to sliding windows alone (Part 3)?
2. Do the additional features (holidays, weekends, etc.) help detect more relevant anomalies?
3. Which features do you think are most important for taxi ridership anomaly detection?
4. What other features could you add to improve detection?

**Your Analysis:**

**Write your analysis here:**

1. Feature engineering vs sliding windows alone:
   - [YOUR ANSWER]

2. Do additional features help?
   - [YOUR ANSWER - Compare detected anomalies with known events]

3. Most important features for taxi ridership:
   - [YOUR ANSWER - Consider day of week, holidays, weather, etc.]

4. Additional features to try:
   - [YOUR IDEAS - Think about external factors affecting taxi demand]

---

# Part 5: Evaluation - Comparing All Approaches

Now that you've applied 4 different temporal preprocessing approaches, let's systematically evaluate and compare them!

## TODO 5.1: Build a Comparison Table

**Your Task:**
Create a comparison table showing which approach detected which known anomaly.

For each approach and algorithm, check if it detected each known event:
- NYC Marathon (2014-11-01)
- Thanksgiving (2014-11-27)
- Christmas (2014-12-25)
- New Year (2015-01-01)
- Blizzard (2015-01-27)

In [ ]:
# TODO: Create a comparison DataFrame
# Hint: Create a dictionary where keys are approach names and values are the detected dates

approaches = {
    'Known Anomalies': nyc_dates,
    # TODO: Add your results here
    # 'KNN - Point Outliers': [...],
    # 'LOF - Point Outliers': [...],
    # 'KNN - Decomposition': [...],
    # etc.
}

# Convert to DataFrame for better visualization
# YOUR CODE HERE

## TODO 5.2: Calculate Detection Rates

**Your Task:**
For each approach, calculate:
1. How many known anomalies were detected (out of 5)?
2. How many total anomalies were detected?
3. What percentage of detected anomalies are known events (precision)?
4. What percentage of known events were detected (recall)?

In [ ]:
# TODO: Calculate detection rates
# Hint: Compare each outlier set with known_outliers.index

def calculate_metrics(detected_outliers, known_dates):
    """
    Calculate precision and recall for anomaly detection.
    
    Args:
        detected_outliers: DataFrame with detected anomalies
        known_dates: List of known anomaly date strings
    
    Returns:
        Dictionary with metrics
    """
    # YOUR CODE HERE
    pass

# Calculate for each approach
# YOUR CODE HERE

## Exercise 5.3: Overall Analysis

**Answer these questions based on your comparison:**

1. Which approach detected the most known anomalies?
2. Which approach had the best balance of precision and recall?
3. Which known events were hardest to detect across all approaches?
4. Which approach would you recommend for production and why?

**Your Analysis:**

**Write your analysis here:**

1. Best approach for detecting known anomalies:
   - [YOUR ANSWER]

2. Best balance of precision and recall:
   - [YOUR ANSWER]

3. Hardest events to detect:
   - [YOUR ANSWER]

4. Recommended approach for production:
   - [YOUR ANSWER]
   - Reasoning: [EXPLAIN YOUR CHOICE]

---

# Challenge Exercise: Apply All 4 Approaches with Isolation Forest

**Challenge:**
You've been using KNN and LOF. Now try applying **Isolation Forest** to all 4 approaches:
1. Point outliers (raw values)
2. Decomposition (residuals)
3. Sliding windows
4. Feature engineering

**Compare:**
- How do Isolation Forest results differ from KNN/LOF?
- Which approach works best with Isolation Forest?
- Is there one "best" combination of algorithm + preprocessing?

In [ ]:
# TODO: Apply Isolation Forest to all 4 approaches
# YOUR CODE HERE

# Approach 1: Raw values
iforest_raw = IForest(contamination=0.03, random_state=42)
# YOUR CODE HERE

# Approach 2: Residuals
# YOUR CODE HERE

# Approach 3: Sliding windows
# YOUR CODE HERE

# Approach 4: Feature engineering
# YOUR CODE HERE

In [ ]:
# TODO: Visualize and compare all 4 Isolation Forest results
# YOUR CODE HERE

**Write your findings:**

1. How does Isolation Forest compare to KNN/LOF?
   - [YOUR ANSWER]

2. Best approach for Isolation Forest:
   - [YOUR ANSWER]

3. Is there one "best" combination?
   - [YOUR ANSWER - Consider that different approaches detect different types of anomalies]

---

# Reflection Questions

Take time to reflect on what you've learned. Write thoughtful answers to these questions:

## 1. When to Use Each Approach

**Question:** Given a new time series anomaly detection problem, how would you decide which temporal preprocessing approach to use?

Consider:
- Type of anomalies you want to detect (point vs pattern)
- Presence of seasonality
- Importance of context
- Computational resources

**Your Answer:**

[WRITE YOUR ANSWER HERE]

## 2. Decomposition vs Sliding Windows

**Question:** Both decomposition and sliding windows try to account for temporal context, but they work very differently. When would you choose one over the other?

**Your Answer:**

[WRITE YOUR ANSWER HERE]

## 3. Feature Engineering Impact

**Question:** Why does adding temporal features (day of week, holidays, etc.) improve anomaly detection? What assumptions are we making?

**Your Answer:**

[WRITE YOUR ANSWER HERE]

## 4. Window Size Selection

**Question:** We used a window size of 7 days. How would you determine the optimal window size for a new dataset? What factors would you consider?

**Your Answer:**

[WRITE YOUR ANSWER HERE]

## 5. Point vs Pattern Anomalies

**Question:** Explain the difference between point anomalies and pattern anomalies. Give a real-world example where each type would be important.

**Your Answer:**

[WRITE YOUR ANSWER HERE]

## 6. Production Deployment Strategy

**Question:** You're deploying a time series anomaly detection system for a ride-sharing company. Describe your deployment strategy:
- Which preprocessing approach(es) would you use?
- Which algorithm(s) would you deploy?
- How would you handle new data?
- How would you validate that the system is working correctly?

**Your Answer:**

[WRITE YOUR ANSWER HERE]

---

# Summary and Key Takeaways

## What You Learned

In this activity, you explored **4 different ways to preprocess time series data** for anomaly detection:

### 1. Point Outlier Detection (Raw Values)
- **What it detects**: Extreme high/low values
- **Pros**: Simple, fast, interpretable
- **Cons**: Ignores temporal context and seasonality
- **Use when**: You care about magnitude outliers regardless of context

### 2. Decomposition-Based Detection (Residuals)
- **What it detects**: Deviations from expected seasonal patterns
- **Pros**: Accounts for trends and seasonality
- **Cons**: Requires stable seasonal patterns
- **Use when**: Your data has clear seasonality and you want context-aware detection

### 3. Sliding Window Detection
- **What it detects**: Unusual temporal sequences and patterns
- **Pros**: Captures recent history and pattern changes
- **Cons**: More complex, requires window size tuning
- **Use when**: You care about temporal patterns and contextual changes

### 4. Feature Engineering Detection
- **What it detects**: Contextual anomalies with domain knowledge
- **Pros**: Most flexible, can encode complex domain knowledge
- **Cons**: Requires feature engineering effort and domain expertise
- **Use when**: You have strong domain knowledge and want maximum control

---

## Critical Insights

1. **The SAME algorithm finds DIFFERENT anomalies** with different preprocessing
2. **No single approach is always best** - it depends on your goals
3. **Combining approaches** (ensemble) often works best in practice
4. **Domain knowledge matters** - especially for feature engineering
5. **Evaluation is crucial** - use known anomalies when available

---

## Next Steps

To deepen your understanding:

1. **Try different window sizes** - how does 3 days vs 14 days affect results?
2. **Experiment with contamination rates** - what if you expect more/fewer anomalies?
3. **Try other algorithms** - How does AutoEncoder perform on time series?
4. **Create an ensemble** - Combine multiple approaches and algorithms
5. **Apply to your own data** - Try these techniques on a different time series

---

## Connection to Activity 1

In **Activity 1**, you learned the fundamental algorithms (KNN, LOF, Isolation Forest, AutoEncoder). In this activity, you learned **how to apply them to time series** through different preprocessing strategies.

**Key difference:**
- Activity 1: "What algorithms can detect anomalies?"
- Activity 2: "How do we prepare time series data for these algorithms?"

Both skills are essential for real-world anomaly detection!

---

**Congratulations on completing Activity 2!**

You now understand how to apply machine learning anomaly detection algorithms to time series data using multiple temporal preprocessing strategies. This is a crucial skill for real-world time series analysis!